# Notebook 03: Quantum Gates and Continuous Rotations

This notebook covers single-qubit transformations and continuous unitary rotations on the Bloch sphere.

---

## Learning Objectives
1. Implement the complete Pauli gate family ($X, Y, Z$).
2. Apply discrete phase gates ($S, T$).
3. Implement continuous parameterized rotations ($R_x, R_y, R_z$).
4. Compute equivalent unitary matrix operators.
5. Verify gate identities using matrix equivalence.


---
## Real-World Applications & Modern Use Cases

Continuous single-qubit rotations are utilized in:
- **Microwave Pulse Control:** Synthesizing targeted radiofrequency and microwave control pulses in superconducting transmon chips.
- **Variational Quantum Optimization:** Dynamically updating gate parameter angles in quantum machine learning models and chemistry eigensolvers.


---
## Section 1: The Pauli Gate Family

The Pauli matrices form an orthogonal basis for all $2 \times 2$ Hermitian matrices:
$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}, \quad Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$


In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

# Construct circuit with Pauli gates
pauli_qc = QuantumCircuit(1)
pauli_qc.x(0)
pauli_qc.y(0)
pauli_qc.z(0)

print("Pauli Gates Circuit:")
print(pauli_qc.draw(output='text'))


Pauli Gates Circuit:
   ┌───┐┌───┐┌───┐
q: ┤ X ├┤ Y ├┤ Z ├
   └───┘└───┘└───┘


---
## Section 2: Discrete Phase Gates: S and T

The S gate performs a $\pi/2$ rotation around the Z-axis ($S = \sqrt{Z}$).
The T gate performs a $\pi/4$ rotation around the Z-axis ($T = \sqrt{S}$).


In [2]:
phase_qc = QuantumCircuit(1)
phase_qc.s(0)
phase_qc.t(0)

print("Phase Gates Circuit:")
print(phase_qc.draw(output='text'))

# Inspect unitary matrix of T gate
t_op = Operator(QuantumCircuit(1))
t_circuit = QuantumCircuit(1)
t_circuit.t(0)
print("\nT Gate Unitary Matrix:")
print(np.round(Operator(t_circuit).data, 3))


Phase Gates Circuit:
   ┌───┐┌───┐
q: ┤ S ├┤ T ├
   └───┘└───┘

T Gate Unitary Matrix:
[[1.   +0.j    0.   +0.j   ]
 [0.   +0.j    0.707+0.707j]]


---
## Section 3: Continuous Parametric Rotations: $R_x, R_y, R_z$

Continuous rotation gates allow arbitrary rotations by angle $\theta$ (in radians):
$$R_x(\theta) = e^{-i \theta X / 2}, \quad R_y(\theta) = e^{-i \theta Y / 2}, \quad R_z(\theta) = e^{-i \theta Z / 2}$$


In [3]:
# Define rotation angles
theta = np.pi / 4
phi = np.pi / 3

rot_qc = QuantumCircuit(1)
rot_qc.rx(theta, 0)
rot_qc.ry(phi, 0)
rot_qc.rz(theta / 2, 0)

print("Parameterized Rotations Circuit:")
print(rot_qc.draw(output='text'))


Parameterized Rotations Circuit:
   ┌─────────┐┌─────────┐┌─────────┐
q: ┤ Rx(π/4) ├┤ Ry(π/3) ├┤ Rz(π/8) ├
   └─────────┘└─────────┘└─────────┘


---
## Section 4: Unitary Matrix Equivalence

Any sequence of quantum gates can be synthesized into a single $2 \times 2$ unitary matrix using `Operator`.


In [4]:
# Compute combined unitary operator
combined_matrix = Operator(rot_qc).data

print("Combined Unitary Matrix (3 decimals):")
print(np.round(combined_matrix, 3))

# Verify unitarity: U * U_dagger = I
identity_check = np.dot(combined_matrix, combined_matrix.conj().T)
print("\nUnitarity Check (U * U^dagger = I):")
print(np.round(identity_check, 4))


Combined Unitary Matrix (3 decimals):
[[ 0.822+0.032j -0.518-0.235j]
 [ 0.518-0.235j  0.822-0.032j]]

Unitarity Check (U * U^dagger = I):
[[1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]


---
## Section 5: Verifying Fundamental Quantum Gate Identities

We verify the famous identity connecting Pauli-X, Pauli-Z, and Hadamard gates:
$$H X H = Z, \quad H Z H = X$$


In [5]:
# Circuit 1: H * Z * H
c1 = QuantumCircuit(1)
c1.h(0)
c1.z(0)
c1.h(0)

# Circuit 2: X
c2 = QuantumCircuit(1)
c2.x(0)

# Compare unitary matrices
op1 = Operator(c1).data
op2 = Operator(c2).data

print("Matrix of H * Z * H:")
print(np.round(op1, 3))
print("\nMatrix of X:")
print(np.round(op2, 3))

print(f"\nAre operators equivalent? {np.allclose(op1, op2)}")


Matrix of H * Z * H:
[[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]

Matrix of X:
[[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]

Are operators equivalent? True


---
## Section 6: Practice Exercises

### Exercise 1: Construct an Arbitrary Superposition
Determine the rotation angle $\theta$ such that $R_y(\theta)|0\rangle$ yields a state with $75\%$ probability of measuring $0$ and $25\%$ probability of measuring $1$. Verify using `Statevector`.


In [6]:
# P(0) = cos(theta / 2)^2 = 0.75
# cos(theta / 2) = sqrt(0.75)
theta_target = 2 * np.arccos(np.sqrt(0.75))

target_qc = QuantumCircuit(1)
target_qc.ry(theta_target, 0)

from qiskit.quantum_info import Statevector
sv_target = Statevector.from_instruction(target_qc)
print(f"Calculated Angle: {theta_target:.4f} rad")
print("Verified Probabilities:", sv_target.probabilities_dict())


Calculated Angle: 1.0472 rad
Verified Probabilities: {'0': 0.7499999999999999, '1': 0.25}
